In [1]:
source("config.r")

In [2]:
library(ggplot2)
library(rpart)      
library(rpart.plot)
library(cowplot)
library(caret)
library(randomForest)
library(tidyr)
library(GGally)
library(corrplot)
library(pROC)
library(nnet)
library(car)
library(dplyr)
library(smotefamily)
library(themis)
library(nnet)
options(warn = -1)

Loading required package: lattice

randomForest 4.7-1.2

Type rfNews() to see new features/changes/bug fixes.


Attaching package: ‘randomForest’


The following object is masked from ‘package:ggplot2’:

    margin


corrplot 0.95 loaded

Type 'citation("pROC")' for a citation.


Attaching package: ‘pROC’


The following objects are masked from ‘package:stats’:

    cov, smooth, var


Loading required package: carData


Attaching package: ‘dplyr’


The following object is masked from ‘package:car’:

    recode


The following object is masked from ‘package:randomForest’:

    combine


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Loading required package: recipes


Attaching package: ‘recipes’


The following object is masked from ‘package:stats’:

    step




In [3]:
df.train <- read.csv(TRAINING_DATA_PATH)

In [4]:
str(df.train)

'data.frame':	440833 obs. of  12 variables:
 $ CustomerID       : int  2 3 4 5 6 8 9 10 11 12 ...
 $ Age              : int  30 65 55 58 23 51 58 55 39 64 ...
 $ Gender           : chr  "Female" "Female" "Female" "Male" ...
 $ Tenure           : int  39 49 14 38 32 33 49 37 12 3 ...
 $ Usage.Frequency  : int  14 1 4 21 20 25 12 8 5 25 ...
 $ Support.Calls    : int  5 10 6 7 5 9 3 4 7 2 ...
 $ Payment.Delay    : int  18 8 18 7 8 26 16 15 4 11 ...
 $ Subscription.Type: chr  "Standard" "Basic" "Basic" "Standard" ...
 $ Contract.Length  : chr  "Annual" "Monthly" "Quarterly" "Monthly" ...
 $ Total.Spend      : num  932 557 185 396 617 129 821 445 969 415 ...
 $ Last.Interaction : int  17 6 3 29 20 8 24 30 13 29 ...
 $ Churn            : int  1 1 1 1 1 1 1 1 1 1 ...


In [5]:
df.train$CustomerID <- NULL
df.train$Gender <- as.factor(df.train$Gender)
df.train$Subscription.Type <- as.factor(df.train$Subscription.Type)
df.train$Contract.Length <- as.factor(df.train$Contract.Length)
df.train$Churn <- as.factor(df.train$Churn)
df.train$Churn.num <- as.numeric(as.character(df.train$Churn))

In [6]:
summary(df.train)

      Age           Gender           Tenure      Usage.Frequency
 Min.   :18.00         :     1   Min.   : 1.00   Min.   : 1.00  
 1st Qu.:29.00   Female:190580   1st Qu.:16.00   1st Qu.: 9.00  
 Median :39.00   Male  :250252   Median :32.00   Median :16.00  
 Mean   :39.37                   Mean   :31.26   Mean   :15.81  
 3rd Qu.:48.00                   3rd Qu.:46.00   3rd Qu.:23.00  
 Max.   :65.00                   Max.   :60.00   Max.   :30.00  
 NA's   :1                       NA's   :1       NA's   :1      
 Support.Calls    Payment.Delay   Subscription.Type  Contract.Length  
 Min.   : 0.000   Min.   : 0.00           :     1            :     1  
 1st Qu.: 1.000   1st Qu.: 6.00   Basic   :143026   Annual   :177198  
 Median : 3.000   Median :12.00   Premium :148678   Monthly  : 87104  
 Mean   : 3.604   Mean   :12.97   Standard:149128   Quarterly:176530  
 3rd Qu.: 6.000   3rd Qu.:19.00                                       
 Max.   :10.000   Max.   :30.00                       

In [10]:
missing.rows <- df.train[!complete.cases(df.train), ]

print(missing.rows, row.names = FALSE)

 Age Gender Tenure Usage.Frequency Support.Calls Payment.Delay
  NA            NA              NA            NA            NA
 Subscription.Type Contract.Length Total.Spend Last.Interaction Churn Churn.num
                                            NA               NA  <NA>        NA


In [11]:
df.train <- df.train[complete.cases(df.train), ]
df.train <- droplevels(df.train)

In [ ]:
summary(df.train)

the data is pretty stable, so we don't need to utilize any imputation method.

In [ ]:
categorical.features <- c("Gender", "Subscription.Type", "Contract.Length")

df.cat.long <- pivot_longer(data = df.train[, c("Churn", categorical.features)],
                            cols = -Churn,
                            names_to = "feature",
                            values_to = "value")

ggplot(df.cat.long, aes(x = value, fill = Churn)) + 
    geom_bar(position = "fill")+
    facet_wrap(~feature, scales = "free_x")+
    labs(title = "Churn Rate by Gender",
         y = "Proportion")

In [ ]:
numeric.features <- c("Age", "Tenure", "Usage.Frequency", "Support.Calls", 
                  "Payment.Delay", "Total.Spend", "Last.Interaction")

df.long <- pivot_longer(data = df.train[, c("Churn", numeric.features)],
                        cols = -Churn,
                        names_to = "feature",
                        values_to = "value")

ggplot(df.long, aes(x = Churn, y = value, fill = Churn)) +
    geom_boxplot(outlier.size = 0.5, alpha = 0.7) + 
    facet_wrap(~ feature, scales = "free_y", ncol = 2) + 
    labs(title = "Numeric Feature Distribution by Churn", 
         x = "Churn", y = "Value") + 
    theme(legend.position = "none")

In [ ]:
cor.data <- df.train[, numeric.features]
cor.data$Churn <- df.train$Churn.num

corrplot(cor(cor.data, use = "complete.obs"),
    method = "color",
    type = "upper",
    tl.cex = 0.8, 
    addCoef.col = "black",
    number.cex = 0.6)

In [ ]:
lm.proxy <- lm(Churn.num ~ . - Churn, data = df.train)
vif.result <- vif(lm.proxy)

vif.data <- data.frame(
    Feature = rownames(vif.result),
    VIF = round(vif.result[, 1], 3)
)

print(vif.data, row.names = FALSE)

In [ ]:
step.result <- stats::step(lm.proxy, direction = "backward")
summary(step.result)

In [ ]:
set.seed(42)
ctrl <- trainControl(method = "cv", number = 5)

cv.model <- train(Churn ~ Age + Gender + Tenure + Usage.Frequency + 
                    Support.Calls + Payment.Delay + Subscription.Type + Contract.Length + 
                    Total.Spend + Last.Interaction,
                data = df.train,
                method = "glm",
                family = "binomial",
                trControl = ctrl, 
                maxit = 300,
                trace = FALSE)

In [ ]:
cv.model

In [12]:
df.test <- read.csv(TESTING_DATA_PATH)
df.test$Churn <- as.factor(df.test$Churn)
missing.rows <- df.test[!complete.cases(df.test), ]
print(missing.rows, row.names = FALSE)

 [1] CustomerID        Age               Gender            Tenure           
 [5] Usage.Frequency   Support.Calls     Payment.Delay     Subscription.Type
 [9] Contract.Length   Total.Spend       Last.Interaction  Churn            
<0 rows> (or 0-length row.names)


In [ ]:
df.test$CustomerID <- NULL

df.test$Gender <- as.factor(df.test$Gender)
df.test$Subscription.Type <- as.factor(df.test$Subscription.Type)
df.test$Contract.Length <- as.factor(df.test$Contract.Length)
df.test$Churn <- as.factor(df.test$Churn)

In [ ]:
str(df.test)
summary(df.test)

In [ ]:
pred <- predict(cv.model, newdata = df.test, type = "raw")
cm <- confusionMatrix(pred, df.test$Churn) 
print(cm$table)

In [ ]:
summary(df.train$Support.Calls)
summary(df.test$Support.Calls)

In [ ]:
summary(df.train$Payment.Delay)
summary(df.test$Payment.Delay)

summary(df.train$Total.Spend)
summary(df.test$Total.Spend)